# 01 — FastF1 first pull (ARIS Phase 1)

**Purpose:** prove the FastF1 cache works end-to-end. Load a real race session from disk (no network round-trip after the first call), inspect the lap dataframe, and produce the first ARIS plot.

**Inputs:** pre-warmed `fastf1_cache/` (see `scripts/prewarm_cache.py`).

**Outputs (this notebook):**
- A `laps` DataFrame for the chosen session, briefly profiled (dtypes, shape, missing-data hot spots).
- A single car-telemetry plot for one driver's fastest lap.

Run top-to-bottom on a fresh kernel. Clear all outputs before committing.

## 1. Imports & cache wiring

The cache lives at `fastf1_cache/` relative to the repo root. The notebook is launched from `notebooks/`, so we resolve the parent and point FastF1 at it. Calling `enable_cache` twice in a session is harmless.

In [ ]:
from pathlib import Path

import fastf1
import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CACHE_DIR = REPO_ROOT / "fastf1_cache"
assert CACHE_DIR.exists(), f"FastF1 cache not found at {CACHE_DIR}. Run scripts/prewarm_cache.py first."
fastf1.Cache.enable_cache(str(CACHE_DIR))

print("fastf1", fastf1.__version__, "| cache:", CACHE_DIR)

## 2. Load a session

Bahrain 2024 Grand Prix — Race. `session.load()` reads from the cache when available; the first call for a fresh session goes to the F1 live-timing API and writes pickle files under `fastf1_cache/<season>/<round>_<event>/`.

Telemetry and laps are needed for the rest of this notebook; weather and race-control messages are skipped to keep the load fast.

In [ ]:
session = fastf1.get_session(2024, "Bahrain", "R")
session.load(telemetry=True, laps=True, weather=False, messages=False)
print(session.event["EventName"], "|", session.name, "|", session.event["EventDate"].date())

## 3. Inspect the lap dataframe

Each row is one driver's one lap. ~31 columns covering lap time, sector times, compound, tyre life, stint, pit-in/out, track status, position, and timestamps. This is the frame the Phase 3 lap-time predictor will train on.

In [ ]:
laps = session.laps
print("shape:", laps.shape)
print("drivers:", sorted(laps["Driver"].unique()))
laps[["Driver", "LapNumber", "LapTime", "Compound", "TyreLife", "Stint"]].head(8)

## 4. Pick Verstappen's fastest lap

`pick_drivers` filters by 3-letter abbreviation; `pick_fastest` returns the single fastest lap as a `Lap` object. `get_car_data()` returns the high-frequency car-channel telemetry (speed, throttle, brake, gear, RPM, DRS) at ~3–10 Hz; `add_distance()` integrates speed over time to give a `Distance` column, which is more useful than wall-clock time for comparing laps.

In [ ]:
ver_lap = laps.pick_drivers("VER").pick_fastest()
tel = ver_lap.get_car_data().add_distance()

lap_time = pd.Timedelta(ver_lap["LapTime"])
m, s, ms = lap_time.components.minutes, lap_time.components.seconds, lap_time.components.milliseconds
print(f"VER fastest: lap {int(ver_lap['LapNumber']):>2} | {m}:{s:02d}.{ms:03d} | "
      f"compound {ver_lap['Compound']} | tyre life {int(ver_lap['TyreLife'])}")
print(f"telemetry: {tel.shape[0]} samples, distance {tel['Distance'].iloc[-1]:.0f} m, "
      f"max speed {tel['Speed'].max():.0f} km/h")

## 5. First plot — speed vs distance

The signature ARIS "hello world" plot. Each dip is a corner; each plateau is a straight. The exit speed and acceleration trace out of each corner is what the Phase 4 counterfactual simulator will eventually perturb.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(tel["Distance"], tel["Speed"], color="#1f77b4", linewidth=1.4)
ax.set_xlabel("Distance (m)")
ax.set_ylabel("Speed (km/h)")
ax.set_title(f"VER fastest race lap — Bahrain 2024 (lap {int(ver_lap['LapNumber'])}, "
             f"{m}:{s:02d}.{ms:03d})")
ax.grid(True, alpha=0.3)
ax.set_xlim(0, tel["Distance"].iloc[-1])
fig.tight_layout()
plt.show()

## Takeaway

If the cache loaded cleanly and the plot rendered, the Phase 1 FastF1 bar is met:
*"can pull a session from cache and draw the first telemetry trace"*.

Next: `02-pandas-basics.ipynb` (pandas refresh) then onward to stint analysis.